# 15 -- Re-búsqueda de hiperparámetros con purge (cuantil relativo de `intermitente`)

Los hiperparámetros vigentes salen de las rondas INV-62/63 (383 corridas), evaluadas **sin purge** y contra los mismos 4 folds con que se reportaba. Aquí se repite la búsqueda con el
protocolo corregido y una política de holdout explícita:

- **Modelo**: LightGBM cuantílico (alpha 0.893) sobre el target relativo `y/base` (`base = nivel_medio_60d·H+1`), 14 features v1, rama `intermitente`.
- **Objetivo de la búsqueda**: costo asimétrico real (Cu=0.25, Co=0.03) por unidad vendida, sobre los **folds de búsqueda (2-4)**, con entrenamiento purgado. El fold 5 **no se toca**
  durante la búsqueda.
- **Búsqueda**: Optuna (TPE, semilla 0), 30 trials; el primer trial es la configuración vigente (`MODELOS_POR_RAMA['intermitente']`), así que la búsqueda no puede empeorarla en 2-4.
  Cada trial se registra en MLflow.
- **Confirmación (una sola vez)**: la mejor configuración se compara con la vigente en el fold 5 (costo pareado con IC95 por bootstrap de productos, cobertura y WAPE del q50).
  Regla: se adopta solo si el IC95 de la mejora de costo en el fold 5 queda sobre 0 **y** el WAPE del q50 no empeora con IC95 < 0.

Limitación: el fold 5 ya fue visto por las rondas anteriores, así que no es un holdout virgen; solo evita contaminación nueva.

Cadena: ... -> `14_calibracion_conformal_estrato` -> **`15_busqueda_hparams_relativo`**

In [1]:
import sys
sys.path.insert(0, '.')
from common_priorizacion import DW, registrar_huella, verificar_huella, purgar_entrenamiento
import evaluar_nivel1 as ev
from exportar_modelos_nivel1 import FEATURES as F1, MODELOS_POR_RAMA, RAMA_A_CONDICION, COSTOS_SUPUESTOS

import json, time, warnings
import numpy as np
import pandas as pd
import lightgbm as lgb
import optuna
warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

pd.set_option('display.width', 230)
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')

EXPERIMENTO = 'EDA-fix-nivel1-15-busqueda-hparams-relativo'
Y, H = 'target_demanda_15d', 15
ALPHA = 0.893
CU, CO = COSTOS_SUPUESTOS['no_perecedero']['Cu'], COSTOS_SUPUESTOS['no_perecedero']['Co']
N_TRIALS, SEMILLA_OPTUNA, N_JOBS = 30, 0, 24
FOLDS_BUSQ, FOLD_HOLDOUT = list(ev.FOLDS_BUSQUEDA), ev.FOLD_HOLDOUT
HP_VIGENTE = dict(MODELOS_POR_RAMA['intermitente']['hparams'])
B, BOOT = 2000, 0

verificar_huella('huella_07.json', ['matriz_as_of.parquet'])
m = pd.read_parquet(f'{DW}/matriz_as_of.parquet')
dr = m.loc[RAMA_A_CONDICION['intermitente'](m)]
cache = {}
for f in FOLDS_BUSQ + [FOLD_HOLDOUT]:
    tr, te = purgar_entrenamiento(dr, f, verbose=False), dr[dr['fold_id'] == f]
    bt, be = tr['nivel_medio_60d'] * H + 1.0, te['nivel_medio_60d'] * H + 1.0
    cache[f] = dict(Xtr=tr[F1], ytr=(tr[Y] / bt).to_numpy(), Xte=te[F1], be=be.to_numpy(), yte=te[Y].to_numpy(),
                    items=te['codigo_item'].astype(str).to_numpy())
    print(f'fold {f}: train={len(tr):,} test={len(te):,}')
print('hiperparámetros vigentes:', HP_VIGENTE)

/home/ddelgadillo@redcorporativa.corpoica.org.co/miniforge3/envs/etl/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026/09/24 20:05:54 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at /home/ddelgadillo@redcorporativa.corpoica.org.co/miniforge3/envs/etl/lib/python3.11/site-packages/mlflow/assistant/skills/instrumenting-with-mlflow-tracing/SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


Huella verificada OK contra huella_07.json.
fold 2: train=20,152 test=68,198
fold 3: train=88,142 test=28,850


fold 4: train=116,468 test=59,600
fold 5: train=175,878 test=76,056
hiperparámetros vigentes: {'n_estimators': 434, 'learning_rate': 0.024519102885452573, 'num_leaves': 78, 'min_child_samples': 15, 'subsample': 0.9970808772067912, 'colsample_bytree': 0.9481956384345176, 'reg_alpha': 0.007967207011040096, 'reg_lambda': 0.005279685262967796}


## 1. Búsqueda (folds 2-4)

In [2]:
def ajusta(hp, f, alpha):
    c = cache[f]
    mod = lgb.LGBMRegressor(objective='quantile', alpha=alpha, subsample_freq=1, random_state=42, verbose=-1, n_jobs=N_JOBS, **hp)
    mod.fit(c['Xtr'], c['ytr'])
    return np.clip(mod.predict(c['Xte']) * c['be'], 0, None)

def costo_por_unidad(hp, folds):
    tot = vend = 0.0
    for f in folds:
        q = ajusta(hp, f, ALPHA)
        tot += ev.costo_asimetrico(cache[f]['yte'], q, CU, CO).sum()
        vend += cache[f]['yte'].sum()
    return tot / vend

def sugerir(trial):
    return dict(n_estimators=trial.suggest_int('n_estimators', 100, 600),
                learning_rate=trial.suggest_float('learning_rate', 0.01, 0.15, log=True),
                num_leaves=trial.suggest_int('num_leaves', 8, 96),
                min_child_samples=trial.suggest_int('min_child_samples', 5, 100),
                subsample=trial.suggest_float('subsample', 0.6, 1.0),
                colsample_bytree=trial.suggest_float('colsample_bytree', 0.5, 1.0),
                reg_alpha=trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
                reg_lambda=trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True))

t0 = time.time()
historial = []
def objetivo(trial):
    hp = sugerir(trial)
    c = costo_por_unidad(hp, FOLDS_BUSQ)
    historial.append(dict(trial=trial.number, costo=c, seg=time.time() - t0, **hp))
    ev.registrar_mlflow(EXPERIMENTO, f'trial_{trial.number:02d}', tags={'fase': 'E', 'notebook': '15', 'tipo': 'trial', 'folds_busqueda': '2,3,4', 'rama_git': 'fix/eda-modelado-nivel1'},
                        params={**hp, 'alpha': ALPHA, 'objetivo': 'costo Cu/Co por unidad, folds 2-4, train con purge', 'semilla_optuna': SEMILLA_OPTUNA},
                        metrics={'costo_por_unidad_f2a4': c})
    print(f'trial {trial.number:2d}: costo/unidad {c:.5f} ({time.time()-t0:,.0f}s)', flush=True)
    return c

estudio = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=SEMILLA_OPTUNA))
estudio.enqueue_trial(HP_VIGENTE)          # el primer trial es la configuración vigente
estudio.optimize(objetivo, n_trials=N_TRIALS)
hist = pd.DataFrame(historial).sort_values('costo').reset_index(drop=True)
print('\nmejores 5 trials (costo por unidad vendida, folds 2-4):'); print(hist.head(5).round(4).to_string(index=False))
costo_vigente = float(pd.DataFrame(historial).iloc[0]['costo'])
mejor = dict(estudio.best_params)
print(f'\nvigente (trial 0): {costo_vigente:.5f} | mejor: {estudio.best_value:.5f} ({(costo_vigente - estudio.best_value) / costo_vigente:+.2%})')

2026/09/24 20:06:04 INFO mlflow.tracking.fluent: Experiment with name 'EDA-fix-nivel1-15-busqueda-hparams-relativo' does not exist. Creating a new experiment.


🏃 View run trial_00 at: http://127.0.0.1:5000/#/experiments/22/runs/d429702f034c4393a906c6e6519aacdd
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/22
trial  0: costo/unidad 0.03788 (10s)


🏃 View run trial_01 at: http://127.0.0.1:5000/#/experiments/22/runs/b346086d35b5449087f8cc8de9a5d143
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/22
trial  1: costo/unidad 0.03832 (18s)


🏃 View run trial_02 at: http://127.0.0.1:5000/#/experiments/22/runs/99793bbaeb3146c994518de615996b12
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/22
trial  2: costo/unidad 0.03805 (31s)


🏃 View run trial_03 at: http://127.0.0.1:5000/#/experiments/22/runs/41a911ab1e444da4a276004a758597b7
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/22
trial  3: costo/unidad 0.03800 (34s)


...

🏃 View run trial_26 at: http://127.0.0.1:5000/#/experiments/22/runs/e916af14484946be8603bceeeec3270e
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/22
trial 26: costo/unidad 0.03740 (151s)


🏃 View run trial_27 at: http://127.0.0.1:5000/#/experiments/22/runs/554ab716a492433fb8ce4af0f1865001
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/22
trial 27: costo/unidad 0.03736 (158s)


🏃 View run trial_28 at: http://127.0.0.1:5000/#/experiments/22/runs/2b71e8ee829648de97485cc997f73ad2
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/22
trial 28: costo/unidad 0.03737 (164s)


🏃 View run trial_29 at: http://127.0.0.1:5000/#/experiments/22/runs/7e8f6933007c40b2bc0b0b30946c736d
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/22
trial 29: costo/unidad 0.03745 (171s)



mejores 5 trials (costo por unidad vendida, folds 2-4):
 trial  costo      seg  n_estimators  learning_rate  num_leaves  min_child_samples  subsample  colsample_bytree  reg_alpha  reg_lambda
    20 0.0373 115.5206           261         0.0105          89                 63     0.7850            0.6970     2.9257      0.0691
    24 0.0373 139.4251           239         0.0120          68                 68     0.7606            0.6653     0.5504      0.1432
    25 0.0374 144.9557           205         0.0180          81                 57     0.7802            0.6784     4.0033      0.0048
    17 0.0374  95.5478           244         0.0117          61                 50     0.8529            0.5879     2.3640      0.0080
    19 0.0374 108.1665           234         0.0114          73                 67     0.8657            0.6566     0.7285      0.0410

vigente (trial 0): 0.03788 | mejor: 0.03734 (+1.43%)


## 2. Confirmación en el fold 5 (una sola vez)

Costo pareado (mejor vs. vigente) con IC95 por bootstrap de productos, cobertura del cuantil y WAPE del q50 (los modelos exportados usan los mismos hiperparámetros para q50 y para el cuantil de negocio).

In [3]:
f = FOLD_HOLDOUT
c5 = cache[f]
q_v, q_b = ajusta(HP_VIGENTE, f, ALPHA), ajusta(mejor, f, ALPHA)
df5 = pd.DataFrame({'codigo_item': c5['items'], 'y': c5['yte']})
cost_v, cost_b = ev.costo_asimetrico(c5['yte'], q_v, CU, CO), ev.costo_asimetrico(c5['yte'], q_b, CU, CO)
r_cost = ev.mejora_perdida_con_ic(df5, cost_b, cost_v, B, BOOT)
p50_v, p50_b = ajusta(HP_VIGENTE, f, 0.5), ajusta(mejor, f, 0.5)
df5[['pv', 'pb']] = np.c_[p50_v, p50_b]
r_w = ev.mejora_con_ic(df5, 'y', 'pb', 'pv', B, BOOT)
res5 = dict(costo_vigente=float(cost_v.sum() / c5['yte'].sum()), costo_mejor=float(cost_b.sum() / c5['yte'].sum()),
            mejora_costo=r_cost['mejora_rel'], ic_costo=[r_cost['ic_bajo'], r_cost['ic_alto']],
            cobertura_vigente=float((c5['yte'] <= q_v).mean()), cobertura_mejor=float((c5['yte'] <= q_b).mean()),
            wape_q50_vigente=r_w['wape_piso'], wape_q50_mejor=r_w['wape_modelo'], mejora_wape_q50=r_w['mejora_rel'], ic_wape_q50=[r_w['ic_bajo'], r_w['ic_alto']])
print(json.dumps({k: (round(v, 4) if isinstance(v, float) else [round(x, 4) for x in v] if isinstance(v, list) else v) for k, v in res5.items()}, indent=1))
adopta = bool(res5['ic_costo'][0] > 0 and res5['mejora_wape_q50'] >= 0)
print('\nDECISIÓN (mejora de costo en fold 5 con IC95 > 0 y WAPE q50 no peor):', 'ADOPTA' if adopta else 'NO adopta (queda la configuración vigente)')

{
 "costo_vigente": 0.0353,
 "costo_mejor": 0.0355,
 "mejora_costo": -0.0056,
 "ic_costo": [
  -0.0127,
  0.0012
 ],
 "cobertura_vigente": 0.8997,
 "cobertura_mejor": 0.9063,
 "wape_q50_vigente": 0.4062,
 "wape_q50_mejor": 0.4088,
 "mejora_wape_q50": -0.0064,
 "ic_wape_q50": [
  -0.014,
  -0.0002
 ]
}

DECISIÓN (mejora de costo en fold 5 con IC95 > 0 y WAPE q50 no peor): NO adopta (queda la configuración vigente)


## 3. Artefactos y registro en MLflow

In [4]:
dec = {'notebook': '15_busqueda_hparams_relativo', 'fecha_generacion': pd.Timestamp.now().isoformat(), 'alpha': ALPHA, 'costos': {'Cu': CU, 'Co': CO},
       'n_trials': N_TRIALS, 'semilla_optuna': SEMILLA_OPTUNA, 'folds_busqueda': FOLDS_BUSQ, 'fold_confirmacion': FOLD_HOLDOUT, 'politica_holdout': ev.POLITICA_HOLDOUT,
       'hparams_vigentes': HP_VIGENTE, 'hparams_mejores': mejor, 'costo_f2a4_vigente': costo_vigente, 'costo_f2a4_mejor': float(estudio.best_value),
       'mejora_f2a4_optimista': (costo_vigente - float(estudio.best_value)) / costo_vigente, 'fold5': res5, 'adopta': adopta}
with open(f'{DW}/decision_15.json', 'w') as fh:
    json.dump(dec, fh, indent=2, ensure_ascii=False)
pd.DataFrame(historial).to_csv(f'{DW}/trials_busqueda_15.csv', index=False)
huella = {'notebook': '15_busqueda_hparams_relativo', 'fecha_generacion': pd.Timestamp.now().isoformat(),
          'entradas': registrar_huella(['matriz_as_of.parquet']), 'salidas': registrar_huella(['decision_15.json', 'trials_busqueda_15.csv'])}
with open(f'{DW}/huella_15.json', 'w') as fh:
    json.dump(huella, fh, indent=2, ensure_ascii=False)
ev.registrar_mlflow(EXPERIMENTO, 'confirmacion_fold5', tags={'fase': 'E', 'notebook': '15', 'tipo': 'confirmacion', 'adopta': str(adopta), 'rama_git': 'fix/eda-modelado-nivel1'},
                    params={**{f'mejor_{k}': v for k, v in mejor.items()}, 'alpha': ALPHA},
                    metrics={'costo_f2a4_vigente': costo_vigente, 'costo_f2a4_mejor': float(estudio.best_value), 'costo_f5_vigente': res5['costo_vigente'],
                             'costo_f5_mejor': res5['costo_mejor'], 'mejora_costo_f5': res5['mejora_costo'], 'ic_costo_f5_bajo': res5['ic_costo'][0],
                             'ic_costo_f5_alto': res5['ic_costo'][1], 'mejora_wape_q50_f5': res5['mejora_wape_q50'],
                             'cobertura_f5_vigente': res5['cobertura_vigente'], 'cobertura_f5_mejor': res5['cobertura_mejor']},
                    artefactos=[f'{DW}/decision_15.json'])
print('guardado: decision_15.json, trials_busqueda_15.csv, huella_15.json; MLflow ok')

🏃 View run confirmacion_fold5 at: http://127.0.0.1:5000/#/experiments/22/runs/900e8407d1cd4cfa89039d71698a1a1d
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/22
guardado: decision_15.json, trials_busqueda_15.csv, huella_15.json; MLflow ok
